# Code mẫu — Huấn luyện trên GPU

Trước khi đẩy một lượt train dài lên GPU, nên chạy thử trên một batch nhỏ. Làm vậy vừa kiểm được cả quy trình, vừa ước lượng được bộ nhớ sẽ cần. Các ví dụ dưới đây cố ý chạy trên CPU, để ai cũng chạy lại được, và tập trung vào ba việc: chọn device, chia batch, và tính trước ngân sách bộ nhớ.

## Ví dụ 1: Kiểm device, rồi cho model học thuộc một batch

Bốn điểm có nhãn theo dấu của tọa độ thứ nhất. Đoạn code luôn chạy trên CPU, nhưng vẫn in ra máy có CUDA hay không. Hai chuyện này khác nhau: máy có GPU không có nghĩa là code đã chạy trên GPU. Cho model học thuộc một nhóm nhỏ là cách kiểm nhanh rằng forward, loss, backward và cập nhật đều thông suốt. Đây không phải là điểm validation.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(307)

cuda_available = torch.cuda.is_available()
print("CUDA available:", cuda_available)
print("CUDA device count:", torch.cuda.device_count())

demo_device = torch.device("cpu")
features = torch.tensor([[-1., -1.], [-1., 1.], [1., -1.], [1., 1.]],
                        device=demo_device)
labels = torch.tensor([0, 0, 1, 1], dtype=torch.long, device=demo_device)
model = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 2)).to(demo_device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

initial = criterion(model(features), labels).item()
for _ in range(60):
    optimizer.zero_grad(set_to_none=True)
    logits = model(features)
    assert logits.shape == (4, 2)
    loss = criterion(logits, labels)
    assert torch.isfinite(loss)
    loss.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all()
               for p in model.parameters())
    optimizer.step()

with torch.no_grad():
    final_logits = model(features)
    final = criterion(final_logits, labels).item()
    accuracy = (final_logits.argmax(dim=1) == labels).float().mean().item()
print("selected device:", demo_device)
print("parameter dtype:", next(model.parameters()).dtype)
print("loss:", round(initial, 6), "->", round(final, 6))
print("one-batch accuracy:", accuracy)
assert next(model.parameters()).device == demo_device
assert accuracy == 1.0

CUDA available: False
CUDA device count: 0
selected device: cpu
parameter dtype: torch.float32
loss: 0.48362 -> 0.000127
one-batch accuracy: 1.0


Máy chạy ví dụ này không có CUDA, và code chạy trên CPU. Trên một máy có GPU, như Kaggle Notebook khi bật GPU, dòng đầu sẽ là `True`, nhưng dòng `selected device` vẫn là `cpu` cho tới khi ta tự đổi `demo_device`. Model, dữ liệu và nhãn phải nằm trên cùng một device. Thiếu một thứ là PyTorch báo lỗi.

Các dòng assert trong vòng lặp kiểm shape của logit, kiểm loss không bị NaN hay vô cùng, và kiểm mọi tham số đều nhận được gradient. Loss giảm gần về 0 và accuracy bằng 1 chỉ cho biết bốn mẫu này đi trót lọt qua cả quy trình. Model chưa được chấm trên mẫu nào mới.

## Ví dụ 2: Chia nhỏ batch và cộng dồn gradient

Khi một batch đầy đủ không vừa bộ nhớ, ta có thể chia nó thành các batch nhỏ, chạy lần lượt, và cộng dồn gradient. Với loss tính trung bình, mỗi batch nhỏ phải được nhân với tỉ lệ `số mẫu của batch nhỏ / tổng số mẫu`. Khi đó gradient cộng dồn sẽ khớp với gradient của batch đầy đủ.

In [ ]:
import copy
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(308)
inputs = torch.tensor([[-2., 0.], [-1., 1.], [0., 2.], [1., 1.],
                      [2., 0.], [3., -1.]])
targets = torch.tensor([[0.], [1.], [2.], [1.], [0.], [-1.]])

base = nn.Linear(2, 1)
full_model = copy.deepcopy(base)
micro_model = copy.deepcopy(base)

full_loss = (full_model(inputs) - targets).square().mean()
full_loss.backward()

total = len(inputs)
for start in range(0, total, 2):
    stop = min(start + 2, total)
    micro_loss = (micro_model(inputs[start:stop]) - targets[start:stop]).square().mean()
    (micro_loss * ((stop - start) / total)).backward()

gradient_gap = max(
    (a.grad - b.grad).abs().max().item()
    for a, b in zip(full_model.parameters(), micro_model.parameters())
)
full_optimizer = torch.optim.SGD(full_model.parameters(), lr=0.1)
micro_optimizer = torch.optim.SGD(micro_model.parameters(), lr=0.1)
full_optimizer.step()
micro_optimizer.step()
parameter_gap = max(
    (a - b).abs().max().item()
    for a, b in zip(full_model.parameters(), micro_model.parameters())
)
print("full-batch MSE:", round(full_loss.item(), 6))
print("microbatch sizes:", [2, 2, 2])
print("max gradient difference:", gradient_gap)
print("max parameter difference after step:", parameter_gap)
assert gradient_gap < 1e-7
assert parameter_gap < 1e-7

full-batch MSE: 1.049236
microbatch sizes: [2, 2, 2]
max gradient difference: 0.0
max parameter difference after step: 0.0


Ba batch nhỏ, mỗi batch hai mẫu, cho đúng cùng gradient và cùng tham số sau một bước như batch sáu mẫu. Mỗi batch nhỏ được nhân với $2/6$. Bỏ quên hệ số này, gradient cộng dồn sẽ gấp ba lần.

Phép kiểm này chỉ xác nhận phần số học. Nếu model có BatchNorm, thống kê được tính trên từng batch nhỏ, nên kết quả không còn giống batch lớn. Dropout và những loss không lấy trung bình đều theo mẫu cũng có thể làm hai cách lệch nhau.

## Ví dụ 3: Ước lượng từng phần của bộ nhớ

Đếm tham số chỉ cho ta biết phần bộ nhớ dành cho trọng số, gradient và trạng thái của optimizer. Activation thì tăng theo kích thước batch. Riêng bảng điểm attention, nếu được tạo ra đầy đủ, còn tăng theo bình phương độ dài chuỗi. Đoạn code dưới đây chỉ tính số byte, không tạo tensor lớn nào, nên chạy được ngay trên máy yếu, trước khi thuê GPU.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(309)
memory_model = nn.Sequential(nn.Linear(6, 10), nn.ReLU(), nn.Linear(10, 3))
parameter_count = sum(p.numel() for p in memory_model.parameters())
weight_bytes = sum(p.numel() * p.element_size() for p in memory_model.parameters())
print("parameters:", parameter_count)
print("weight bytes:", weight_bytes)
print("FP32 Adam weight+grad+2 moments estimate:", 16 * parameter_count, "bytes")

for batch in [8, 16]:
    hidden_activation_bytes = batch * 10 * 4
    print("batch", batch, "hidden activation:", hidden_activation_bytes, "bytes")

def mib(byte_count):
    return round(byte_count / 2**20, 3)

for length in [128, 256]:
    score_bytes = 2 * 4 * length * length * 4  # B × số head × L × L × 4 byte FP32.
    print("attention score B=2,H=4,L=", length, ":", mib(score_bytes), "MiB")
assert parameter_count == 103
assert 16 * parameter_count == 1648

parameters: 103
weight bytes: 412
FP32 Adam weight+grad+2 moments estimate: 1648 bytes
batch 8 hidden activation: 320 bytes
batch 16 hidden activation: 640 bytes
attention score B=2,H=4,L= 128 : 0.5 MiB
attention score B=2,H=4,L= 256 : 2.0 MiB


Mạng có 103 tham số, mỗi tham số FP32 chiếm 4 byte, nên trọng số chiếm 412 byte. Với Adam, mỗi tham số cần thêm một gradient và hai moment, tổng cộng 16 byte, tức 1648 byte. Con số này không đổi khi batch tăng từ 8 lên 16. Activation của lớp ẩn thì tăng gấp đôi theo batch.

Bảng điểm attention tăng gấp bốn khi độ dài chuỗi tăng gấp đôi, từ 128 lên 256, vì bảng có hai trục cùng dài `L`. Đây mới là ước lượng trên giấy. Nó chưa tính các activation khác, vùng nhớ tạm của các phép toán, và phần dư của bộ cấp phát. Bộ nhớ đỉnh và thời gian chạy thật phải đo trên chính máy có GPU.